In [ ]:
"""
Day 9 — train and evaluate classical + XGBoost models for AQI forecasting.
KAGGLE-STANDALONE VERSION — all config/utils inlined, no repo imports needed.

Models trained per horizon (24h, 48h, 72h):
    - Naive baseline   (predict tomorrow = today's AQI)
    - Ridge regression (with StandardScaler + alpha grid search)
    - Random Forest    (small grid: n_estimators, max_depth)
    - XGBoost          (RandomizedSearchCV, 20 trials)
    - XGBoost p10      (quantile, no HPT)
    - XGBoost p90      (quantile, no HPT)

Validation:
    - Chronological 80/20 split
    - 5-fold walk-forward CV on the 80% training pool for HPT
    - Metrics reported on the 20% holdout (touched once at the end)

Hard rule (Draft 6, MOM): training reads from Hopsworks only.
Models are NOT pushed to the registry today.

Setup on Kaggle:
    1. Add-ons -> Secrets -> add HOPSWORKS_API_KEY
    2. pip install hopsworks xgboost (usually xgboost is preinstalled)
    3. Run all cells / `python train_classical_kaggle.py`
"""

import argparse
import sys
import time
from typing import Dict, Tuple

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# ── Install deps (Kaggle base image doesn't have hopsworks) ────────────────
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hopsworks"], check=True)

# hopsworks pulls in protobuf<5, but Kaggle's preinstalled tensorflow needs
# protobuf>=5.28. If we `import tensorflow` now it will crash with
# "cannot import name 'runtime_version' from 'google.protobuf'". Force
# protobuf back up BEFORE importing tensorflow (nothing has imported it
# yet, so no kernel restart is needed) — this fixes the tf import without
# touching hopsworks' own functionality, which doesn't care about the
# protobuf version bump.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "protobuf>=5.28,<6"],
    check=True,
)

# hopsworks also pulls cryptography up to a version newer than the
# pyOpenSSL already on Kaggle supports. tensorflow optionally imports
# googleapiclient -> oauth2client -> pyOpenSSL (for a GCE cluster
# resolver we never use), and that import chain breaks with
# "AttributeError: module 'lib' has no attribute 'GEN_EMAIL'" on the old
# pyOpenSSL + new cryptography combo. Upgrade pyOpenSSL to match before
# tensorflow gets imported.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pyopenssl"],
    check=True,
)

# ========================================================================
# Config (inlined — originally config.config)
# ========================================================================

FEATURE_GROUP_NAME = "aqi_features"
FEATURE_GROUP_VERSION = 1
HORIZONS = ["24h", "48h", "72h"]

TRAIN_TEST_SPLIT = 0.8     # chronological holdout: 80% train pool / 20% holdout
CV_N_SPLITS = 5            # walk-forward folds used for Ridge/RF grid search
HPT_CV_FOLDS = 5           # walk-forward folds used for XGBoost RandomizedSearchCV
HPT_N_ITER = 20            # XGBoost RandomizedSearchCV trials

# Columns that are never features: identifiers, targets, and the
# full-target-coverage flag. Everything else in the feature group is
# used as a feature 
NON_FEATURE_COLS = {
    "timestamp", "has_target",
    "target_aqi_24h", "target_aqi_48h", "target_aqi_72h",
}

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("train_classical_kaggle")


# ========================================================================
# Hopsworks connection (inlined — originally utils.hopsworks_client)
# ========================================================================

def get_feature_store():
    import hopsworks

    # Prefer Kaggle Secrets; fall back to env var if you set one manually.
    api_key = None
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("HOPSWORKS_API_KEY")
    except Exception:
        api_key = os.environ.get("HOPSWORKS_API_KEY")

    if not api_key:
        raise RuntimeError(
            "No Hopsworks API key found. Add it under Add-ons -> Secrets as "
            "HOPSWORKS_API_KEY, or set the HOPSWORKS_API_KEY environment variable."
        )

    project = hopsworks.login(api_key_value=api_key)
    return project.get_feature_store()



# ========================================================================
# CV utility (inlined — originally utils.cv)
# ========================================================================

def chronological_holdout_split(df: pd.DataFrame, train_fraction: float):
    split_idx = int(len(df) * train_fraction)
    return df.iloc[:split_idx].reset_index(drop=True), df.iloc[split_idx:].reset_index(drop=True)


def walk_forward_splits(n_rows: int, n_splits: int):
    """Expanding-window walk-forward CV splits (no shuffling)."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    return list(tscv.split(np.arange(n_rows)))


# ========================================================================
# Metrics (inlined — originally utils.metrics)
# ========================================================================

def regression_metrics(y_true, y_pred) -> Dict[str, float]:
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

def print_metrics_by_horizon(results: Dict[str, Dict[str, Dict[str, float]]]) -> None:
    for model_name, per_horizon in results.items():
        print(f"\n{model_name}:")
        for h in HORIZONS:
            if h not in per_horizon:
                continue
            m = per_horizon[h]
            print(f"  {h:>4s}  RMSE={m['rmse']:.2f}  MAE={m['mae']:.2f}  R2={m['r2']:.3f}")


# ========================================================================
# Data loading & prep
# ========================================================================

def load_training_data() -> pd.DataFrame:
    logger.info("Reading aqi_features from Hopsworks")
    fs = get_feature_store()
    fg = fs.get_feature_group(FEATURE_GROUP_NAME, version=FEATURE_GROUP_VERSION)
    df = fg.read()

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)

    logger.info("Total rows: %d", len(df))
    df = df[df["has_target"] == 1].copy()
    logger.info("Rows with full targets: %d", len(df))
    return df

def get_feature_columns(df: pd.DataFrame) -> list:
    """All columns except targets/identifiers"""
    return [c for c in df.columns if c not in NON_FEATURE_COLS]

def split_xy(df: pd.DataFrame, feature_cols: list) -> Tuple[pd.DataFrame, Dict[str, pd.Series]]:
    X = df[feature_cols].copy()
    y = {h: df[f"target_aqi_{h}"] for h in HORIZONS if f"target_aqi_{h}" in df.columns}
    return X, y

# ========================================================================
# Model builders (per horizon)
# ========================================================================

def fit_naive(_X_train, _y_train, X_holdout) -> np.ndarray:
    return X_holdout["aqi"].to_numpy()


def fit_ridge(X_train, y_train, X_holdout, quick: bool = False):
    pipe = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge())])
    alpha_grid = [0.1, 1.0, 10.0] if quick else [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
    param_grid = {"ridge__alpha": alpha_grid}

    cv_splits = walk_forward_splits(len(X_train), n_splits=CV_N_SPLITS)
    search = GridSearchCV(pipe, param_grid=param_grid, cv=cv_splits,
                           scoring="neg_root_mean_squared_error", n_jobs=-1)
    search.fit(X_train, y_train)
    logger.info("    best alpha=%s | CV RMSE=%.3f",
                search.best_params_["ridge__alpha"], -search.best_score_)
    return search.best_estimator_, search.best_estimator_.predict(X_holdout)


def fit_rf(X_train, y_train, X_holdout, quick: bool = False):
    if quick:
        param_grid = {"n_estimators": [50], "max_depth": [10]}
    else:
        param_grid = {"n_estimators": [100, 200], "max_depth": [10, 20, None]}

    cv_splits = walk_forward_splits(len(X_train), n_splits=CV_N_SPLITS)
    search = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=-1),
                           param_grid=param_grid, cv=cv_splits,
                           scoring="neg_root_mean_squared_error", n_jobs=-1)
    search.fit(X_train, y_train)
    logger.info("    best params=%s | CV RMSE=%.3f", search.best_params_, -search.best_score_)
    return search.best_estimator_, search.best_estimator_.predict(X_holdout)


def fit_xgb_main(X_train, y_train, X_holdout, quick: bool = False):
    n_iter = 5 if quick else HPT_N_ITER
    param_dist = {
        "n_estimators": [200, 400, 600, 800],
        "max_depth": [4, 6, 8, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "min_child_weight": [1, 3, 5, 7],
    }
    cv_splits = walk_forward_splits(len(X_train), n_splits=HPT_CV_FOLDS)
    search = RandomizedSearchCV(
        XGBRegressor(objective="reg:squarederror", random_state=42,
                     n_jobs=1, tree_method="hist"),
        param_distributions=param_dist, n_iter=n_iter, cv=cv_splits,
        scoring="neg_root_mean_squared_error", n_jobs=-1, random_state=42, verbose=0,
    )
    search.fit(X_train, y_train)
    logger.info("    best params=%s | CV RMSE=%.3f", search.best_params_, -search.best_score_)
    return search.best_estimator_, search.best_estimator_.predict(X_holdout)

def run_shap_analysis(xgb_models: Dict[str, "XGBRegressor"], X_holdout: pd.DataFrame,
                       feature_cols: list, top_n: int = 15) -> None:
    """Print mean |SHAP value| per feature, per horizon, using the
    already-tuned XGBoost model for that horizon. Saves a CSV per
    horizon too so you can inspect/sort/plot it yourself later."""
    import shap
 
    for h, model in xgb_models.items():
        logger.info("Computing SHAP values for horizon %s", h)
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_holdout)
 
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        importance = pd.DataFrame({
            "feature": feature_cols,
            "mean_abs_shap": mean_abs_shap,
        }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
 
        out_path = f"shap_importance_{h}.csv"
        importance.to_csv(out_path, index=False)
        logger.info("Saved %s", out_path)
 
        print(f"\nTop {top_n} features by mean |SHAP| — horizon {h}:")
        print(importance.head(top_n).to_string(index=False))

# ========================================================================
# Main orchestration
# ========================================================================

def train_all_models(quick: bool = False) -> None:
    df = load_training_data()
 
    train_df, holdout_df = chronological_holdout_split(df, train_fraction=TRAIN_TEST_SPLIT)
    logger.info("Train pool: %d rows | Holdout: %d rows", len(train_df), len(holdout_df))
    logger.info("Train ends: %s | Holdout starts: %s",
                train_df["timestamp"].iloc[-1], holdout_df["timestamp"].iloc[0])
 
    feature_cols = get_feature_columns(df)
    logger.info("Using %d feature columns: %s", len(feature_cols), feature_cols)
 
    X_train, y_train_dict = split_xy(train_df, feature_cols)
    X_holdout, y_holdout_dict = split_xy(holdout_df, feature_cols)
 
    feature_medians = X_train.median()
    X_train = X_train.fillna(feature_medians)
    X_holdout = X_holdout.fillna(feature_medians)
 
    results: Dict[str, Dict[str, Dict[str, float]]] = {
        "Naive": {}, "Ridge": {}, "RF": {}, "XGBoost": {},
    }
    xgb_models: Dict[str, XGBRegressor] = {}  # kept per horizon for the SHAP step below
 
    for h in HORIZONS:
        logger.info("=== Horizon: %s ===", h)
        y_train = y_train_dict[h]
        y_holdout = y_holdout_dict[h]
 
        logger.info("  [Naive] no fit needed")
        pred = fit_naive(X_train, y_train, X_holdout)
        results["Naive"][h] = regression_metrics(y_holdout, pred)
 
        logger.info("  [Ridge] fitting with grid search")
        t0 = time.time()
        _, pred = fit_ridge(X_train, y_train, X_holdout, quick=quick)
        logger.info("    fit time: %.1fs", time.time() - t0)
        results["Ridge"][h] = regression_metrics(y_holdout, pred)
 
        logger.info("  [RF] fitting with grid search")
        t0 = time.time()
        _, pred = fit_rf(X_train, y_train, X_holdout, quick=quick)
        logger.info("    fit time: %.1fs", time.time() - t0)
        results["RF"][h] = regression_metrics(y_holdout, pred)
 
        logger.info("  [XGBoost] fitting with RandomizedSearchCV")
        t0 = time.time()
        xgb_model, pred = fit_xgb_main(X_train, y_train, X_holdout, quick=quick)
        logger.info("    fit time: %.1fs", time.time() - t0)
        results["XGBoost"][h] = regression_metrics(y_holdout, pred)
        xgb_models[h] = xgb_model
 
    print("\n" + "=" * 50)
    print("HOLDOUT METRICS — main models")
    print("=" * 50)
    print_metrics_by_horizon(results)
 
    # ============================================================
    # SHAP — feature importance per horizon, on the tuned XGBoost
    # model. This does NOT prune anything automatically; it just
    # prints/saves rankings so you can pick features yourself.
    # ============================================================
    run_shap_analysis(xgb_models, X_holdout, feature_cols)

def _parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Train classical + XGBoost models for AQI forecasting (Kaggle-standalone)"
    )
    parser.add_argument("--quick", action="store_true",
                        help="Use smaller hyperparameter grids for testing (~1-2 min)")
    # argparse chokes on Kaggle/Jupyter's injected -f flag — ignore unknowns
    args, _ = parser.parse_known_args()
    return args


def main() -> None:
    args = _parse_args()
    if args.quick:
        logger.info("QUICK MODE: small grids, 5 trials for XGBoost")
    try:
        t0 = time.time()
        train_all_models(quick=args.quick)
        logger.info("Total training time: %.1fs", time.time() - t0)
    except Exception:
        logger.exception("Training failed")
        sys.exit(1)

if __name__ == "__main__":
    main()